In [ ]:
import sys, os, warnings
from pathlib import Path
import numpy as np


NOTEBOOK_DIR = Path().resolve()
SRC_DIR      = NOTEBOOK_DIR.parent / "tsforge"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# ── Imports ───────────────────────────────────────────────────────────────────
from metrics.stability_metrics import forecast_percentage_change, excess_volatility

warnings.filterwarnings("ignore")
print("imports OK")


## TEST - sFPC and sEV

In [ ]:
B, T, H, C = 1, 10, 5, 2
quantiles = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
Q = len(quantiles)

pattern = np.array([1, 2, 3, 4, 5], dtype=float)
preds_single = np.stack([np.roll(pattern, -t) for t in range(T)])  # (T, H)

# add C and Q dims
preds = np.stack([preds_single, preds_single], axis=-1)[None]       # (1, T, H, C)
y = preds.copy()                                                  # (1, T, H, C)
preds = preds[..., np.newaxis].repeat(Q, axis=-1)                   # (1, T, H, C, Q)
mask = np.ones((B, T, H, C))

fpc = forecast_percentage_change(preds=preds[..., 4], symmetric=True, mask=mask) # sFPC is a pointwise metric
sev = excess_volatility(targets=y, preds=preds, quantiles=quantiles, scaling=True, mask=mask)

assert np.isclose(fpc, 0.0, atol=1e-9), "sFPC should be zero."
assert np.isclose(sev, 0.0, atol=1e-9), "sEV should be zero."
print("\n✓ TEST PASSED")

In [ ]:
B, T, H, C = 1, 10, 5, 2
quantiles = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
Q = len(quantiles)

y = np.ones((B, T, H, C))
preds = np.ones((B, T, H, C, Q))
mask = np.ones((B, T, H, C))

fpc = forecast_percentage_change(preds=preds[..., 4], symmetric=True, mask=mask) # sFPC is a pointwise metric
sev = excess_volatility(targets=y, preds=preds, quantiles=quantiles, scaling=True, mask=mask)

assert np.isclose(fpc, 0.0, atol=1e-9), "sFPC should be zero."
assert np.isclose(sev, 0.0, atol=1e-9), "sEV should be zero."
print("\n✓ TEST PASSED")